# Solution 05 — deps that carry an HTTP client

`WeatherDeps` holds a sync `httpx.Client`; the tools reach it through `ctx.deps.client`, so they stay plain
`def`. `raise_for_status()` plus `except httpx.HTTPError` turns every failed request (bad status, timeout, no
network) into a `ModelRetry`. The stretch (`unit`) is included. Needs network access.

In [ ]:
from learningdantic import setup

MODEL = setup()

In [ ]:
from dataclasses import dataclass

import httpx
from pydantic_ai import Agent, ModelRetry, RunContext


@dataclass
class WeatherDeps:
    client: httpx.Client
    unit: str = "celsius"


weather_agent = Agent(
    MODEL,
    deps_type=WeatherDeps,
    instructions=(
        "You compare the current weather in cities. For each city, call geocode, then call "
        "current_temp with its latitude and longitude. Never guess a temperature."
    ),
)


@weather_agent.tool
def geocode(ctx: RunContext[WeatherDeps], city: str) -> dict:
    """Find the latitude and longitude of a city. Pass just the city name, like "Paris"."""
    try:
        response = ctx.deps.client.get(
            "https://geocoding-api.open-meteo.com/v1/search", params={"name": city, "count": 1}
        )
        response.raise_for_status()
    except httpx.HTTPError as e:
        raise ModelRetry(f"Geocoding request failed ({e}). Try again.") from e
    results = response.json().get("results") or []
    if not results:
        raise ModelRetry(f"No place called {city!r}. Pass just the city name, like 'Paris'.")
    hit = results[0]
    return {"name": hit["name"], "latitude": hit["latitude"], "longitude": hit["longitude"]}


@weather_agent.tool
def current_temp(ctx: RunContext[WeatherDeps], latitude: float, longitude: float) -> str:
    """Get the current temperature at a location."""
    try:
        response = ctx.deps.client.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": latitude,
                "longitude": longitude,
                "current": "temperature_2m",
                "temperature_unit": ctx.deps.unit,
            },
        )
        response.raise_for_status()
    except httpx.HTTPError as e:
        raise ModelRetry(f"Weather request failed ({e}). Try again.") from e
    data = response.json()
    return f"{data['current']['temperature_2m']} {data['current_units']['temperature_2m']}"

In [ ]:
with httpx.Client(timeout=10) as client:
    result = await weather_agent.run(
        "Is it warmer right now: Paris or Tokyo? Give both temperatures.",
        deps=WeatherDeps(client),
    )
print(result.output)

In [ ]:
# Offline check: with no network, the tool must raise ModelRetry, not crash.
# A tool is a plain function, so call it with a stand-in ctx that only has .deps.
from types import SimpleNamespace


def no_network(request):
    raise httpx.ConnectError("no network")


retried = False
with httpx.Client(transport=httpx.MockTransport(no_network)) as dead_client:
    try:
        geocode(SimpleNamespace(deps=WeatherDeps(dead_client)), "Paris")
    except ModelRetry as e:
        retried = True
        print("ModelRetry:", e)

In [ ]:
# Every tool the model called during the run, in order.
calls = [p.tool_name for m in result.all_messages() for p in m.parts if p.part_kind == "tool-call"]
print(calls)

assert calls.count("geocode") >= 2, calls  # both cities were looked up
assert calls.count("current_temp") >= 2, calls  # both temperatures came from the API
assert "Paris" in result.output and "Tokyo" in result.output, result.output
assert retried, "a failed request should become a ModelRetry"
print("✅ all checks passed")